### IMPORTs

In [1]:
from dotenv import load_dotenv, find_dotenv
import pandas as pd
import os

### Carregando dados de Vendas

In [2]:
# 1. Carrega as variáveis do arquivo .env automaticamente, onde quer que ele esteja
load_dotenv(find_dotenv())

# 2. Puxa o caminho da pasta de CSVs configurado no .env
diretorio_csv = os.getenv('CAMINHO_DIRETORIO_CSV')

# 3. Cria o caminho completo (Pasta + Nome do Arquivo)
caminho_vendas_csv = os.path.join(diretorio_csv, "vendas_carros.csv")

# 4. Lê o CSV usando o caminho completo
df_vendas = pd.read_csv(caminho_vendas_csv)

# 5. Exibe o DataFrame na tela
display(df_vendas)

,id_venda,id_loja,modelo_carro,marca,valor,data_venda,id_vendedor
0,1,105,Taycan,Porsche,467705.77,24-08-2025,9
1,2,103,BRZ,Subaru,219397.40,2025-03-07,5
2,3,102,M3,BMW,272253.32,03-01-2026,4
3,4,102,S2000,Honda,449832.70,11/04/2025,3
4,5,105,GT-R,Nissan,212980.87,2025-03-14,10
...,...,...,...,...,...,...,...
195,196,104,911 Carrera,Porsche,445491.01,2025-10-19,8
196,197,102,Lancer Evolution,Mitsubishi,NaN,13/01/2025,4
197,198,101,BRZ,Subaru,270222.01,17/01/2026,2
198,199,102,NSX,Honda,426460.87,31/01/2026,4


### Verificando NULLs

In [3]:
print("Verificando quantidade de NULLs:")
print(df_vendas.isnull().sum())

Verificando quantidade de NULLs:
id_venda         0
id_loja          0
modelo_carro     0
marca            0
valor           14
data_venda       0
id_vendedor      0
dtype: int64


### Padronizando DATAS

In [4]:
df_vendas['data_venda'] = pd.to_datetime(df_vendas['data_venda'], format='mixed', dayfirst=True)
display(df_vendas)

,id_venda,id_loja,modelo_carro,marca,valor,data_venda,id_vendedor
0,1,105,Taycan,Porsche,467705.77,2025-08-24,9
1,2,103,BRZ,Subaru,219397.40,2025-07-03,5
2,3,102,M3,BMW,272253.32,2026-01-03,4
3,4,102,S2000,Honda,449832.70,2025-04-11,3
4,5,105,GT-R,Nissan,212980.87,2025-03-14,10
...,...,...,...,...,...,...,...
195,196,104,911 Carrera,Porsche,445491.01,2025-10-19,8
196,197,102,Lancer Evolution,Mitsubishi,NaN,2025-01-13,4
197,198,101,BRZ,Subaru,270222.01,2026-01-17,2
198,199,102,NSX,Honda,426460.87,2026-01-31,4


### Tratando colunas `valor` para os valores nulls

In [5]:
df_vendas['valor'] = df_vendas['valor'].fillna(
    df_vendas.groupby('marca')['valor'].transform('median')
)

print("Verificando quantidade de NULLs depois do tratamento:")
print(df_vendas.isnull().sum())

Verificando quantidade de NULLs depois do tratamento:
id_venda        0
id_loja         0
modelo_carro    0
marca           0
valor           0
data_venda      0
id_vendedor     0
dtype: int64


### Tratando casas decimais

In [6]:
df_vendas['valor'] = df_vendas['valor'].round(2)

### criando Ticket

In [7]:
# criando nova coluna
df_vendas['categoria_ticket'] = pd.cut(
    df_vendas['valor'],
    bins=[0, 150000, 300000, float('inf')], 
    labels=['Entrada', 'Intermediário', 'Premium']
)

In [8]:
display(df_vendas)

,id_venda,id_loja,modelo_carro,marca,valor,data_venda,id_vendedor,categoria_ticket
0,1,105,Taycan,Porsche,467705.77,2025-08-24,9,Premium
1,2,103,BRZ,Subaru,219397.40,2025-07-03,5,Intermediário
2,3,102,M3,BMW,272253.32,2026-01-03,4,Intermediário
3,4,102,S2000,Honda,449832.70,2025-04-11,3,Premium
4,5,105,GT-R,Nissan,212980.87,2025-03-14,10,Intermediário
...,...,...,...,...,...,...,...,...
195,196,104,911 Carrera,Porsche,445491.01,2025-10-19,8,Premium
196,197,102,Lancer Evolution,Mitsubishi,276524.60,2025-01-13,4,Intermediário
197,198,101,BRZ,Subaru,270222.01,2026-01-17,2,Intermediário
198,199,102,NSX,Honda,426460.87,2026-01-31,4,Premium


### Criando PARQUET

In [9]:
# 1. Carrega as variáveis do arquivo .env automaticamente
load_dotenv(find_dotenv())

# 2. Puxa o caminho da pasta de Parquets configurado no .env
diretorio_base = os.getenv('CAMINHO_DIRETORIO_PARQUET')

# 3. Cria o caminho completo (Pasta + Nome do Arquivo)
caminho_completo_vendas = os.path.join(diretorio_base, "vendas_carros_prata.parquet")

# 4. Salva o DataFrame de vendas no diretório especificado
df_vendas.to_parquet(caminho_completo_vendas, engine="pyarrow", index=False)

# 5. Print seguro: confirma a criação sem expor as pastas do seu computador
print("✅ Arquivo 'vendas_carros_prata.parquet' gerado com sucesso no diretório configurado!")

✅ Arquivo 'vendas_carros_prata.parquet' gerado com sucesso no diretório configurado!
